In [ ]:
import os
import shutil
import yaml
import random
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from ultralytics import YOLO, RTDETR

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
# Dataset Options: 
# "v1" -> mahyeks dataset (70/15/15 split)
# "v2" -> revised dataset (already split)
DATASET_CHOICE = "v1"
# Model Options: "yolov8s.pt", "yolo26s.pt", "rtdetr-l.pt"
MODEL_CHOICE = "rtdetr-l.pt"

IMG_SIZE = 640
BATCH_SIZE = 16
TUNE_EPOCHS = 30
TUNE_ITERATIONS = 10
FINAL_EPOCHS = 100

V1_BASE = Path("../input/datasets/mahyeks/multi-class-strawberry-ripeness-detection-dataset/all")
V2_BASE = Path("../input/datasets/samatsauranbek/multi-class-strawberry-ripeness-detection-v2")
WORKING_DIR = Path("/kaggle/working/yolo_dataset")

In [ ]:
def get_present_classes(label_path):
    if not os.path.exists(label_path): return []
    with open(label_path, 'r') as f:
        return list(set([int(line.split()[0]) for line in f.readlines()]))

if DATASET_CHOICE == "v1":
    images_dir, labels_dir = V1_BASE / "images", V1_BASE / "labels"
    images = sorted([f for f in os.listdir(images_dir) if f.endswith(('.jpg', '.png'))])
    
    y_matrix = np.zeros((len(images), 3))
    for i, img_name in enumerate(images):
        classes = get_present_classes(labels_dir / (img_name.rsplit('.', 1)[0] + '.txt'))
        for c in classes: y_matrix[i, c] = 1
            
    X = np.array(images)
    msss_train = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
    train_idx, temp_idx = next(msss_train.split(X, y_matrix))
    X_train, y_temp, X_temp = X[train_idx], y_matrix[temp_idx], X[temp_idx]
    
    msss_test = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
    val_idx, test_idx = next(msss_test.split(X_temp, y_temp))
    X_val, X_test = X_temp[val_idx], X_temp[test_idx]

    for split in ['train', 'val', 'test']:
        os.makedirs(WORKING_DIR / 'images' / split, exist_ok=True)
        os.makedirs(WORKING_DIR / 'labels' / split, exist_ok=True)
        
    def populate(file_list, split_name):
        for img in file_list:
            lbl = img.rsplit('.', 1)[0] + '.txt'
            if (images_dir / img).exists(): shutil.copy(images_dir / img, WORKING_DIR / 'images' / split_name / img)
            if (labels_dir / lbl).exists(): shutil.copy(labels_dir / lbl, WORKING_DIR / 'labels' / split_name / lbl)
            
    populate(X_train, 'train')
    populate(X_val, 'val')
    populate(X_test, 'test')
    
    base_data_path = WORKING_DIR

elif DATASET_CHOICE == "v2":
    base_data_path = V2_BASE

yaml_tune = {
    'path': str(base_data_path.resolve()),
    'train': 'images/train' if DATASET_CHOICE == "v1" else 'train/images',
    'val': 'images/val' if DATASET_CHOICE == "v1" else 'val/images',
    'test': 'images/test' if DATASET_CHOICE == "v1" else 'test/images',
    'nc': 3,
    'names': ['Fullripe', 'Semiripe', 'Unripe']
}

yaml_final = {
    'path': str(base_data_path.resolve()),
    'train': [
        'images/train' if DATASET_CHOICE == "v1" else 'train/images',
        'images/val' if DATASET_CHOICE == "v1" else 'val/images'
    ],
    'val': 'images/test' if DATASET_CHOICE == "v1" else 'test/images',
    'nc': 3,
    'names': ['Fullripe', 'Semiripe', 'Unripe']
}

yaml_tune_path = Path("/kaggle/working/data_tune.yaml")
yaml_final_path = Path("/kaggle/working/data_final.yaml")

with open(yaml_tune_path, 'w') as f: yaml.dump(yaml_tune, f, sort_keys=False)
with open(yaml_final_path, 'w') as f: yaml.dump(yaml_final, f, sort_keys=False)

In [ ]:
if "rtdetr" in MODEL_CHOICE.lower():
    tune_model = RTDETR(MODEL_CHOICE)
else:
    tune_model = YOLO(MODEL_CHOICE)

tune_model.tune(
    data=str(yaml_tune_path),
    epochs=TUNE_EPOCHS,
    iterations=TUNE_ITERATIONS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=0,
    project='/kaggle/working/runs/tune',
    name=f'{MODEL_CHOICE.split(".")[0]}_tuning'
)

best_hyp_path = f'/kaggle/working/runs/tune/{MODEL_CHOICE.split(".")[0]}_tuning/best_hyperparameters.yaml'

In [ ]:
if "rtdetr" in MODEL_CHOICE.lower():
    final_model = RTDETR(MODEL_CHOICE)
else:
    final_model = YOLO(MODEL_CHOICE)

final_results = final_model.train(
    data=str(yaml_final_path),     
    epochs=FINAL_EPOCHS,                
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=0,
    cfg=best_hyp_path if os.path.exists(best_hyp_path) else None,
    patience=20,
    project='/kaggle/working/runs/final',
    name=f'{MODEL_CHOICE.split(".")[0]}_final'
)

In [ ]:
best_weights = Path(f"/kaggle/working/runs/final/{MODEL_CHOICE.split('.')[0]}_final/weights/best.pt")

if "rtdetr" in MODEL_CHOICE.lower():
    eval_model = RTDETR(str(best_weights))
else:
    eval_model = YOLO(str(best_weights))

test_img_dir = Path(yaml_tune['path']) / yaml_tune['test']
sample_images = sorted([p for p in test_img_dir.iterdir() if p.suffix.lower() in {'.jpg', '.png'}])[:8]

pred_results = eval_model.predict(
    source=[str(p) for p in sample_images],
    imgsz=IMG_SIZE,
    conf=0.25,
    iou=0.50,
    save=True,
    project="/kaggle/working/predictions",
    name=f"{MODEL_CHOICE.split('.')[0]}_preds",
    exist_ok=True,
    device=0,
)

PRED_DIR = Path(f"/kaggle/working/predictions/{MODEL_CHOICE.split('.')[0]}_preds")
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, img_path in zip(axes.flatten(), sorted(PRED_DIR.glob("*"))[:8]):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.axis('off')
    
plt.tight_layout()
plt.show()

In [ ]:
from datasets import Dataset, DatasetDict
from datasets import Image as HFImage
from PIL import Image
from transformers import RTDetrV2ForObjectDetection, RTDetrImageProcessor
import torch

# 1. Convert YOLO txt annotations to Hugging Face absolute bounding boxes
def yolo_to_hf_format(image_dir, label_dir):
    data = {"image_id": [], "image": [], "width": [], "height": [], "objects": []}
    image_paths = list(Path(image_dir).glob("*.jpg")) + list(Path(image_dir).glob("*.png"))
    
    for idx, img_path in enumerate(image_paths):
        label_path = Path(label_dir) / f"{img_path.stem}.txt"
        if not label_path.exists(): continue
            
        with Image.open(img_path) as img: w, h = img.size
        objects = {"id": [], "area": [], "bbox": [], "category": []}
        
        with open(label_path, "r") as f:
            for obj_id, line in enumerate(f.readlines()):
                parts = line.strip().split()
                if len(parts) < 5: continue 
                
                class_id = int(parts[0])
                coords = list(map(float, parts[1:]))
                
                # Handle polygons or standard boxes to find extremities
                x_coords, y_coords = coords[0::2], coords[1::2] 
                x_min_norm, x_max_norm = min(x_coords), max(x_coords)
                y_min_norm, y_max_norm = min(y_coords), max(y_coords)
                
                abs_x_min, abs_y_min = x_min_norm * w, y_min_norm * h
                abs_w, abs_h = (x_max_norm - x_min_norm) * w, (y_max_norm - y_min_norm) * h
                
                objects["id"].append(obj_id)
                objects["area"].append(abs_w * abs_h)
                objects["bbox"].append([abs_x_min, abs_y_min, abs_w, abs_h])
                objects["category"].append(class_id)
        
        data["image_id"].append(idx)
        data["image"].append(str(img_path))
        data["width"].append(w)
        data["height"].append(h)
        data["objects"].append(objects)
    return data

print("Converting YOLO dataset to Hugging Face format...")
hf_dataset = DatasetDict({
    "train": Dataset.from_dict(yolo_to_hf_format(WORKING_DIR / 'images/train', WORKING_DIR / 'labels/train')),
    "validation": Dataset.from_dict(yolo_to_hf_format(WORKING_DIR / 'images/val', WORKING_DIR / 'labels/val')),
    "test": Dataset.from_dict(yolo_to_hf_format(WORKING_DIR / 'images/test', WORKING_DIR / 'labels/test'))
}).cast_column("image", HFImage())

# 2. Load Processor and Model
id2label = {0: 'Fullripe', 1: 'Semiripe', 2: 'Unripe'}
label2id = {'Fullripe': 0, 'Semiripe': 1, 'Unripe': 2}
MODEL_ID = "PekingU/rtdetr_v2_r18vd"

processor = RTDetrImageProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model_hf = RTDetrV2ForObjectDetection.from_pretrained(
    MODEL_ID, id2label=id2label, label2id=label2id,
    ignore_mismatched_sizes=True, trust_remote_code=True
)
print("HF Dataset and Model Ready!")

In [ ]:
import numpy as np
import albumentations as A

# Albumentations setup
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.SafeRotate(limit=15, p=0.4), 
    A.ColorJitter(brightness=0.0, contrast=0.0, saturation=0.0, hue=0.03, p=0.5),
], bbox_params=A.BboxParams(format='coco', label_fields=['category']))

def transform_batch(batch, is_training=False):
    images, annotations = [], []
    for i in range(len(batch["image"])):
        image_np = np.array(batch["image"][i].convert("RGB"))
        bboxes, categories = batch["objects"][i]["bbox"], batch["objects"][i]["category"]
        
        if is_training and len(bboxes) > 0:
            augmented = train_transform(image=image_np, bboxes=bboxes, category=categories)
            image_np, bboxes, categories = augmented['image'], augmented['bboxes'], augmented['category']
            
        images.append(image_np)
        formatted_annotation = [
            {"category_id": cat, "bbox": box, "area": box[2] * box[3]} 
            for cat, box in zip(categories, bboxes)
        ]
        annotations.append({"image_id": batch["image_id"][i], "annotations": formatted_annotation})

    inputs = processor(images=images, annotations=annotations, return_tensors="pt")
    return {"pixel_values": inputs["pixel_values"], "labels": inputs["labels"]}

def collate_fn(batch):
    pixel_values = torch.stack([item["pixel_values"] for item in batch])
    labels = [item["labels"] for item in batch]
    return {"pixel_values": pixel_values, "labels": labels}

# Apply transforms
from datasets import concatenate_datasets
combined_train = concatenate_datasets([hf_dataset["train"], hf_dataset["validation"]])

active_train_dataset = combined_train.with_transform(lambda b: transform_batch(b, is_training=True))
active_test_dataset = hf_dataset["test"].with_transform(lambda b: transform_batch(b, is_training=False))

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

output_dir = "/kaggle/working/hf_rtdetr_final"

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=8,   
    per_device_eval_batch_size=8,
    num_train_epochs=30,             
    learning_rate=5e-5,              
    weight_decay=1e-4,
    warmup_ratio=0.1,                
    eval_strategy="epoch",           
    save_strategy="epoch",
    save_total_limit=2,              
    logging_steps=20,
    remove_unused_columns=False,     
    dataloader_num_workers=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss", 
    greater_is_better=False,         
    fp16=True,                       
)

trainer = Trainer(
    model=model_hf,
    args=training_args,
    train_dataset=active_train_dataset,
    eval_dataset=active_test_dataset,  # Using test set for final eval tracking
    data_collator=collate_fn,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)] 
)

print("Starting Hugging Face Trainer...")
trainer.train()

trainer.save_model(f"{output_dir}_model")
processor.save_pretrained(f"{output_dir}_model")
print(f"Training Complete! Saved to {output_dir}_model")

In [ ]:
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import pandas as pd
from torchmetrics.detection.mean_ap import MeanAveragePrecision

metric = MeanAveragePrecision(box_format='xyxy', iou_type='bbox', class_metrics=True)
test_dataloader = DataLoader(active_test_dataset, batch_size=8, collate_fn=collate_fn, num_workers=2)

model_hf.eval()
print("Running test set inference...")
with torch.no_grad():
    for batch in tqdm(test_dataloader):
        pixel_values = batch["pixel_values"].to(model_hf.device)
        labels = [{k: v.to(model_hf.device) for k, v in t.items()} for t in batch["labels"]]

        outputs = model_hf(pixel_values=pixel_values, labels=labels)
        
        target_sizes = torch.tensor([[pixel_values.shape[2], pixel_values.shape[3]]] * len(labels))
        results = processor.post_process_object_detection(outputs, target_sizes=target_sizes, threshold=0.25)

        formatted_preds, formatted_targets = [], []
        img_h, img_w = pixel_values.shape[2], pixel_values.shape[3]

        for result, target in zip(results, labels):
            formatted_preds.append({
                "boxes": result["boxes"].cpu(),
                "scores": result["scores"].cpu(),
                "labels": result["labels"].cpu()
            })
            
            target_boxes = target["boxes"].cpu()
            cx, cy = target_boxes[:, 0] * img_w, target_boxes[:, 1] * img_h
            w, h = target_boxes[:, 2] * img_w, target_boxes[:, 3] * img_h
            
            target_boxes_xyxy = torch.zeros_like(target_boxes)
            target_boxes_xyxy[:, 0] = cx - (w / 2)
            target_boxes_xyxy[:, 1] = cy - (h / 2)
            target_boxes_xyxy[:, 2] = cx + (w / 2)
            target_boxes_xyxy[:, 3] = cy + (h / 2)
            
            formatted_targets.append({
                "boxes": target_boxes_xyxy,
                "labels": target["class_labels"].cpu()
            })

        metric.update(formatted_preds, formatted_targets)

results_dict = metric.compute()

df_class = pd.DataFrame({
    'Class': ['Fullripe', 'Semiripe', 'Unripe'],
    'mAP (50-95)': results_dict['map_per_class'].tolist(),
    'Recall (max 100)': results_dict['mar_100_per_class'].tolist() 
})

print("\n" + "="*50)
print(f"OVERALL mAP (50-95): {results_dict['map'].item():.4f}")
print(f"OVERALL mAP@50     : {results_dict['map_50'].item():.4f}")
print("="*50)
print(df_class.to_string(index=False))